# Módulo 7 · Silver y Gold (arquitectura medallón)

⏱ **10 min** · Slide: 28

```
BRONZE  bronze_episodios     crudo, anidado, con origen
   │  aplanar · tipos correctos · sin duplicados
   ▼
SILVER  silver_episodios     limpio, una fila por episodio
   │  agregar
   ▼
GOLD    gold_resumen_red · gold_emisiones_dia     listo para tableros
```

En la práctica `01_ELT` este paso se hacía con pandas. Aquí lo hacemos **100 % con PySpark**, así corre distribuido.

| Paso | Qué haces |
|---|---|
| 1 | Construir Silver |
| 2 | Construir Gold |
| 3 | Ver el shuffle de una agregación |
| 4 | Reconstruir desde Bronze |

**Requiere:** módulo 5.

In [ ]:
%run ./00_Setup

## Paso 1 · Silver: aplanar, tipar y deduplicar

Cada paso de limpieza queda explícito en la función.

In [ ]:
def build_silver():
    s = spark.table(BRONZE)

    silver = (s.select(
            F.col("id").alias("episode_id"),
            F.col("name").alias("episode_name"),
            "season",
            F.col("number").alias("episode_number"),
            F.col("type").alias("episode_type"),
            F.expr("try_cast(airdate AS DATE)").alias("airdate"),          # texto → fecha (NULL si no es válida)
            F.expr("try_cast(airstamp AS TIMESTAMP)").alias("airstamp"),   # texto → timestamp
            F.col("runtime").alias("runtime_min"),
            F.col("rating.average").alias("episode_rating"),
            F.col("_embedded.show.id").alias("show_id"),
            F.col("_embedded.show.name").alias("show_name"),
            F.col("_embedded.show.type").alias("show_type"),
            F.col("_embedded.show.language").alias("show_language"),
            F.col("_embedded.show.status").alias("show_status"),
            F.concat_ws(",", F.col("_embedded.show.genres")).alias("show_genres"),
            F.coalesce(F.col("_embedded.show.network.name"),
                       F.col("_embedded.show.webChannel.name"),
                       F.lit("(sin red)")).alias("network_name"),
            F.when(F.col("_embedded.show.network.id").isNotNull(), "broadcast")
             .when(F.col("_embedded.show.webChannel.id").isNotNull(), "web")
             .otherwise("desconocida").alias("network_type"),
            "archivo_origen")
        .filter(F.col("episode_id").isNotNull())      # sin clave no hay episodio
        .dropDuplicates(["episode_id"]))              # un episodio = una fila

    (silver.write.format("delta").mode("overwrite")
        .option("overwriteSchema", True)
        .saveAsTable(SILVER))

In [ ]:
build_silver()

filas_bronze = spark.table(BRONZE).count()
filas_silver = spark.table(SILVER).count()
print(f"Bronze: {filas_bronze} filas → Silver: {filas_silver} filas  (eliminadas: {filas_bronze - filas_silver})")

In [ ]:
display(spark.table(SILVER).limit(10))

In [ ]:
spark.table(SILVER).printSchema()

👀 **Qué observar:** columnas **planas**, `airdate` ya es `date` y `airstamp` es `timestamp` (en Bronze eran texto).
Si Bronze y Silver tienen el mismo conteo, no había duplicados en esta carga.

## Paso 2 · Gold: agregados listos para consumir

Dos tablas para dos preguntas de negocio:
1. ¿Qué redes/plataformas emiten más contenido?
2. ¿Cómo se reparten las emisiones por día?

In [ ]:
def build_gold():
    s = spark.table(SILVER)

    resumen_red = (s.groupBy("network_name", "network_type")
        .agg(F.count("*").alias("episodios"),
             F.countDistinct("show_id").alias("series"),
             F.round(F.avg("episode_rating"), 2).alias("rating_promedio"),
             F.round(F.avg("runtime_min"), 1).alias("minutos_promedio"))
        .orderBy(F.col("episodios").desc()))

    emisiones_dia = (s.groupBy("airdate")
        .agg(F.count("*").alias("episodios"),
             F.sum("runtime_min").alias("minutos_totales"))
        .withColumn("dia_semana", F.date_format("airdate", "EEEE"))
        .orderBy("airdate"))

    resumen_red.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(GOLD_RED)
    emisiones_dia.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(GOLD_DIA)

In [ ]:
build_gold()
display(spark.table(GOLD_RED).limit(10))

In [ ]:
display(spark.table(GOLD_DIA))

👀 **Qué observar:** de miles de filas pasamos a unas pocas decenas. Gold es pequeño, rápido y responde una pregunta concreta.
Prueba convertir `gold_emisiones_dia` en un gráfico de barras con el botón **+** sobre la tabla.

## Paso 3 · El costo de agregar: shuffle
Cada `groupBy` de Gold reordena datos entre workers (módulo 3). Búscalo en el plan:

In [ ]:
spark.table(SILVER).groupBy("network_name").count().explain()

👀 **Qué observar:** de nuevo `Exchange`. Con millones de filas, este es el paso que más cuesta.

## Paso 4 · "Si algo falla aguas abajo, reconstruyes desde Bronze"

Simulamos un desastre: borramos Silver y Gold. Bronze no se toca.

In [ ]:
for t in (SILVER, GOLD_RED, GOLD_DIA):
    spark.sql(f"DROP TABLE IF EXISTS {t}")
print("💥 Silver y Gold eliminadas")

In [ ]:
build_silver()
build_gold()

print("✔ Reconstruidas desde Bronze")
print("  Silver     :", spark.table(SILVER).count(), "filas")
print("  Gold (red) :", spark.table(GOLD_RED).count(), "filas")
print("  Gold (día) :", spark.table(GOLD_DIA).count(), "filas")

👀 **Qué observar:** todo vuelve a existir con los mismos números. Por eso Bronze se guarda crudo y nunca se modifica.

## ✔ Checkpoint

- [ ] `silver_episodios`, `gold_resumen_red` y `gold_emisiones_dia` existen
- [ ] Silver tiene tipos correctos y una fila por episodio
- [ ] Reconstruí Silver y Gold desde Bronze

**Reto opcional:** crea `gold_series` (una fila por serie con su idioma, estado y nº de episodios). Ya tienes todas las columnas en Silver.

**Siguiente:** `08_Lakehouse_y_Cierre`